In [ ]:
#

In [ ]:
!gcloud auth login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=7FGs0wCgulkf9m9EWWmFGUaZeBlqUG&prompt=consent&token_usage=remote&access_type=offline&code_challenge=jaWt6dC3VQSIa1mNHn2_jeGYm1ifbcJEiqTYe3m3Z2I&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AXlqoi5yXBBAKy8ebvTBFqgY-GsK3mlM-38nxiPvfhkOzoU1r6KD8ZIqlzNbuUZ_Oa1GsQ

You are now logged in as [ghurbal.akbar@gmail.com].
Your current projec

In [ ]:
from google.colab import userdata

# Private values live here in the notebook, like the API keys — never in the repo.
GCP_DATASET_PATH = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning/v2_arabmaqamrock_dataset"   # v2 folder prefix (renamed from dataset/ 2026-10-10)
GCP_DATASET_ZIP = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning/v3_arabmaqamrock_dataset.zip"   # v3 (438, verbatim): setup.sh unzips it into /content/yue2_dataset when set (zip wins)
GCP_BACKUP_BASE = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning"

# pron_dataset is only for the AR-only pron run; NOT needed for quran_ahh_r8.
# GCP_PRON_DATASET_PATH = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning/pron_dataset"

# The AHH run's dataset (quran_ahh_r8) — NOT needed for the maqamrock run. Uncomment
# only to also pull the ~3.69 GiB AHH zip (setup.sh runs it as an extra job).
# GCP_AHH_DATASET_ZIP = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning/quran_ahh_dataset.zip"

with open('/root/.secrets.env', 'w') as f:
    f.write(f'export DEEPSEEK_API_KEY="{userdata.get("DEEPSEEK_API_KEY")}"\n')
    f.write(f'export HF_TOKEN="{userdata.get("HF_TOKEN")}"\n')
    f.write(f'export GCP_DATASET_PATH="{GCP_DATASET_PATH}"\n')
    f.write(f'export GCP_DATASET_ZIP="{GCP_DATASET_ZIP}"\n')
    # f.write(f'export GCP_PRON_DATASET_PATH="{GCP_PRON_DATASET_PATH}"\n')
    # f.write(f'export GCP_AHH_DATASET_ZIP="{GCP_AHH_DATASET_ZIP}"\n')
    f.write(f'export GCP_BACKUP_BASE="{GCP_BACKUP_BASE}"\n')

marker = 'source /root/.secrets.env'
with open('/root/.bashrc') as f:
    already_wired = marker in f.read()
if not already_wired:
    with open('/root/.bashrc', 'a') as f:
        f.write(f'\n{marker}\n')

print("Secrets staged. Open the Colab terminal now.")


Secrets staged. Open the Colab terminal now.


In [ ]:
%%bash
cd /content && rm -f vscode_cli.tar.gz
curl -Lk 'https://vscode.download.prss.microsoft.com/dbazure/download/stable/07f806f999227108933c2e30515b26eecc1fda74/vscode_cli_alpine_x64_cli.tar.gz' --output vscode_cli.tar.gz
file vscode_cli.tar.gz          # want: "gzip compressed data"
tar -xzf vscode_cli.tar.gz

vscode_cli.tar.gz: gzip compressed data, from Unix, original size modulo 2^32 34928640


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12.3M  100 12.3M    0     0   184M      0 --:--:-- --:--:-- --:--:--  186M


In [ ]:
%%bash
curl -fsSL https://opencode.ai/v2/install | bash &


Installing opencode version: 2.0.23
Successfully added opencode to $PATH in /root/.bashrc

                                 ▄     
█▀▀█ █▀▀█ █▀▀█ █▀▀▄ █▀▀▀ █▀▀█ █▀▀█ █▀▀█
█░░█ █░░█ █▀▀▀ █░░█ █░░░ █░░█ █░░█ █▀▀▀
▀▀▀▀ █▀▀▀ ▀▀▀▀ ▀  ▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀


OpenCode includes free models, to start:

cd <project>  # Open directory
opencode      # Run command

For more information visit https://opencode.ai/v2/docs




######################################################################## 100.0%


## Training setup (fresh VM)

```bash
# 0. Load the staged env (bucket paths). The cell above wrote /root/.secrets.env.
. /root/.secrets.env

# 1. The redirect target must exist BEFORE bash opens it. /content/logs
#    does not exist on a fresh Colab VM, and a failed redirect kills the
#    whole line silently (setup.sh never starts).
mkdir -p /content/logs

# 2. Clone (skip if already present, so re-pasting the block is safe).
[ -d maqamrock-yue2-lora-finetuning ] || git clone https://github.com/akbargherbal/maqamrock-yue2-lora-finetuning.git
cd maqamrock-yue2-lora-finetuning
# 2b. The v3 maqamrock run (config, docs, v3 dataset support) lives on main.
git checkout main

# 3. Launch setup DETACHED. It runs its jobs in PARALLEL, including the v3
#    dataset zip pull (GCP_DATASET_ZIP is set), so the ~2.5 GiB download
#    proceeds while you authenticate the tunnel below.
#    Logs: /content/logs/setup.log (summary) + /content/logs/dataset_zip.log (the pull).
setsid nohup bash bootstrap/setup.sh > /content/logs/setup.log 2>&1 < /dev/null &
disown

# 4. Confirm it actually started (should print a PID).
pgrep -af 'bootstrap/setup.sh'

# 5. Authenticate the tunnel in the FOREGROUND; Do this terminal (Interactive).
# /content/code tunnel
```

When setup finishes, its summary must show `[ok]   dataset: 438 tracks in /content/yue2_dataset`
— otherwise read `/content/logs/dataset_zip.log`. Then start the run per
`docs/START.md`.


## Inference setup (fresh VM)

```bash
# 1. The redirect target must exist BEFORE bash opens it. /content/logs
#    does not exist on a fresh Colab VM, and a failed redirect kills the
#    whole line silently (setup.sh never starts).
mkdir -p /content/logs

# 2. Clone (skip if already present, so re-pasting the block is safe).
[ -d maqamrock-yue2-lora-finetuning ] || git clone https://github.com/akbargherbal/maqamrock-yue2-lora-finetuning.git
cd maqamrock-yue2-lora-finetuning

# 3. Launch INFERENCE setup DETACHED (--inference clones audio.cpp and
#    pre-warms the bf16 GGUF + VAE; it skips the dataset and ai-toolkit).
setsid nohup bash bootstrap/setup.sh --inference > /content/logs/setup.log 2>&1 < /dev/null &
disown

# 4. Confirm it actually started before moving on (should print a PID).
pgrep -af 'bootstrap/setup.sh'

# 5. Authenticate the tunnel in the FOREGROUND; Do this terminal (Interactive).
# /content/code tunnel
```

After setup finishes, build audio.cpp once (it is only cloned, never built)
and pull the converted LoRA + runner scripts from the run's GCS prefix.


```js
function ClickConnect() {
  console.log("Checking connection status...");
  const colabButton = document.querySelector("colab-connect-button");
  if (colabButton && colabButton.shadowRoot) {
    const connectBtn = colabButton.shadowRoot.querySelector("#connect");
    if (connectBtn) {
      connectBtn.click();
      console.log("Connect button clicked ✅");
    } else {
      console.log("Connect button not found ❌");
    }
  } else {
    console.log("colab-connect-button element not found ❌");
  }
}
setInterval(ClickConnect, 60000); // Clicks every 60 seconds

```

```bash
mkdir -p /content/logs
cd /content
. ~/.bashrc
# ai-toolkit SOURCE only — the agent needs to read toolkit/lora_special.py,
# not run training. Skip its requirements.txt entirely (that's the CUDA pin).
git clone https://github.com/ostris/ai-toolkit.git &

# This project's repo
[ -d maqamrock-yue2-lora-finetuning ] || git clone https://github.com/akbargherbal/maqamrock-yue2-lora-finetuning.git
cd maqamrock-yue2-lora-finetuning
git checkout pron-lora-ar-only

# CPU-only torch + safetensors — all the merge tool actually needs to run
pip install -q --no-input torch --index-url https://download.pytorch.org/whl/cpu
pip install -q --no-input safetensors

# The existing converter script + its current outputs (for the invariant sha256 test),
# ~133 MiB, not the multi-GB inference bundle
mkdir -p /content/converter/out
gsutil -m rsync -r "$GCP_BACKUP_BASE/audiocpp_inference/converter" /content/ &converter/out

/content/code tunnel
```

In [ ]:
%%bash
. ~/.bashrc

In [ ]:
%%bash
# 0. Load the staged env (bucket paths). The cell above wrote /root/.secrets.env.
. /root/.secrets.env

# 1. The redirect target must exist BEFORE bash opens it. /content/logs
#    does not exist on a fresh Colab VM, and a failed redirect kills the
#    whole line silently (setup.sh never starts).
mkdir -p /content/logs

# 2. Clone (skip if already present, so re-pasting the block is safe).
[ -d maqamrock-yue2-lora-finetuning ] || git clone https://github.com/akbargherbal/maqamrock-yue2-lora-finetuning.git
cd maqamrock-yue2-lora-finetuning
# 2b. The v3 maqamrock run (config, docs, v3 dataset support) lives on main.
git checkout main

# 3. Launch setup DETACHED. It runs its jobs in PARALLEL, including the v3
#    dataset zip pull (GCP_DATASET_ZIP is set), so the ~2.5 GiB download
#    proceeds while you authenticate the tunnel below.
#    Logs: /content/logs/setup.log (summary) + /content/logs/dataset_zip.log (the pull).
setsid nohup bash bootstrap/setup.sh > /content/logs/setup.log 2>&1 < /dev/null &
disown

# 4. Confirm it actually started (should print a PID).
pgrep -af 'bootstrap/setup.sh'

# 5. Authenticate the tunnel in the FOREGROUND; Do this terminal (Interactive).
# /content/code tunnel


branch 'pron-lora-long-aya' set up to track 'origin/pron-lora-long-aya'.
15831 bash bootstrap/setup.sh


Cloning into 'maqamrock-yue2-lora-finetuning'...
Switched to a new branch 'pron-lora-long-aya'


In [ ]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2026-10-06 00:53:51'